# Daily helicorder plots from an SDS archive (ObsPy `SDSClient`)

This notebook reads continuous waveform data for **AV.REF.00.\*** (all channels) from an SDS archive using ObsPy’s filesystem client, then generates **daily helicorder (dayplot) plots**.

**Default SDS root:** `/Volumes/classdata/SDS_Alaska` (macOS Samba mount in this class)

SDS layout expected (SeisComP/SLArchive convention):

```
<SDS_ROOT>/<YEAR>/<NET>/<STA>/<CHAN>.<TYPE>/<NET>.<STA>.<LOC>.<CHAN>.<TYPE>.<YEAR>.<JJJ>
```


In [ ]:
import os
from pathlib import Path
from obspy import UTCDateTime
from obspy.clients.filesystem.sds import Client as SDSClient

# -------------------------
# User settings
# -------------------------
SDS_ROOT = Path("/Volumes/classdata/SDS_Alaska")   # <-- default for this course
NET = "AV"
STA = "REF"
LOC = ""
CHA = "*"   # all channels

# Date range (inclusive start, exclusive end)
t0 = UTCDateTime("2009-03-20T00:00:00")
t1 = UTCDateTime("2009-03-24T00:00:00")

# Output folder for PNGs (kept OUTSIDE the SDS tree)
outdir = Path.home() / "DATA" / "helicorders_AV_REF_00"
outdir.mkdir(parents=True, exist_ok=True)

print("SDS_ROOT:", SDS_ROOT)
print("Output plots:", outdir)
print("Date range:", t0, "to", t1)


In [ ]:
# Create SDS client
client = SDSClient(str(SDS_ROOT))

# Quick sanity check: does the SDS root exist?
if not SDS_ROOT.exists():
    raise FileNotFoundError(
        f"SDS_ROOT does not exist: {SDS_ROOT}\n"
        "If you are not on macOS, change SDS_ROOT to your mounted path."
    )

print("SDS client ready.")


## Helper: read one day and make helicorder plots

We:
1. Read one day (00:00–24:00) for AV.REF.00.\*
2. Merge segments (keeps gaps)
3. Plot one helicorder per channel as a PNG


In [ ]:
from obspy import Stream

def read_one_day(net: str, sta: str, loc: str, cha: str, day_start: UTCDateTime) -> Stream:
    """Read one UTC day of data for a station from SDS."""
    day_end = day_start + 24 * 3600
    st = client.get_waveforms(net, sta, loc, cha, day_start, day_end)
    # Merge segments per channel; keep gaps as gaps
    try:
        st.merge(method=1, fill_value=None)
    except Exception as e:
        print(f"⚠️ merge failed for {net}.{sta}.{loc}.{cha} on {day_start.date}: {e}")
    return st

def plot_dayplots(st: Stream, net: str, sta: str, loc: str, day_start: UTCDateTime):
    """Write helicorder PNGs for each channel in the stream."""
    if len(st) == 0:
        print(f"  (no data)")
        return

    day_tag = day_start.strftime("%Y-%m-%d")
    # Ensure stable ordering: by channel code then start time
    st = st.copy()
    st.sort(keys=["channel", "starttime"])

    for tr in st.select(channel="*Z"):
        tr.plot(
            type="dayplot",
            interval=60,            # seconds per line
            right_vertical_labels=False,
            one_tick_per_line=True,
            #show=False,
            show=True,
            #outfile=str(outfile),
        )




In [ ]:
# Loop through days, read, and plot
day = UTCDateTime(t0.date)  # snap to 00:00 of the start date
while day < t1:
    print(f"\n=== {day.date} ===")
    st_day = read_one_day(NET, STA, LOC, CHA, day)
    print(f"  downloaded traces: {len(st_day)}")
    plot_dayplots(st_day, NET, STA, LOC, day)
    day += 24 * 3600

print("\nDone.")


## Notes

- **Empty days** usually mean no data exist for that day/channel in the SDS tree (or your `SDS_ROOT` is wrong).
- If plots look “broken”, it’s usually **gaps** (normal) or a **timing issue** in the archive.
- Adjust `interval=` to change how many seconds each helicorder line represents.
